In [ ]:
import pandas as pd
from geopy.distance import geodesic
import numpy as np

# 1. Read CSV safely
df = pd.read_csv("equities.csv", low_memory=False)

# 2. Parse date column
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# 3. Filter for 1 December 2021
target_date = pd.Timestamp("2021-12-01")
df_filtered = df[df["date"] == target_date]

# 4. Keep only one observation per RIC
df_unique = df_filtered.drop_duplicates(subset="RIC", keep="first")

# 5. Drop ONLY 'Unnamed: 0' and all columns starting with 'msci_'
df_unique = df_unique.drop(columns=[col for col in df_unique.columns 
                                    if col == "Unnamed: 0" or col.startswith("msci_")])

# 6. Compute distance (in km) from firm HQ (lat, long) to invasion point (49.872, 36.935)
def compute_distance(row):
    try:
        firm_coords = (row["lat"], row["long"])
        invasion_coords = (49.872, 36.935)
        return geodesic(firm_coords, invasion_coords).kilometers
    except:
        return None

df_unique["dist_invasion"] = df_unique.apply(compute_distance, axis=1)

# 7. Binary treatment: 1 if dist_ukr < 1000 km
df_unique["treat_ukr"] = (df_unique["dist_ukr"] < 1000).astype(int)

# 8. Binary dummy for invasion proximity
df_unique["treat_invasion"] = (df_unique["dist_invasion"] < 1000).astype(int)

# 9. Define first- and second-degree neighbor sets (excluding Russia)
first_deg_iso2 = {"BY", "PL", "SK", "HU", "RO", "MD"}  # Belarus, Poland, Slovakia, Hungary, Romania, Moldova
first_deg_iso3 = {"BLR", "POL", "SVK", "HUN", "ROU", "MDA"}

second_deg_iso2 = {"DE", "CZ", "LT", "AT", "RS", "HR", "SI", "BG", "LV"}
second_deg_iso3 = {"DEU", "CZE", "LTU", "AUT", "SRB", "HRV", "SVN", "BGR", "LVA"}

# 10. First-order neighbor dummy
df_unique["nbr_1"] = (
    df_unique["ctriso2"].isin(first_deg_iso2) |
    df_unique["ctriso3"].isin(first_deg_iso3)
).astype(int)

# 11. First- or second-order neighbor dummy
df_unique["nbr_1_or_2"] = (
    df_unique["ctriso2"].isin(first_deg_iso2.union(second_deg_iso2)) |
    df_unique["ctriso3"].isin(first_deg_iso3.union(second_deg_iso3))
).astype(int)

# 12. --- Continuous treatment variables based on dist_ukr ---

# a) Inverse-distance
df_unique["treat_ukr_inv"] = 1 / (1 + df_unique["dist_ukr"])

# b) Normalized distance (rescaled 0–1)
max_dist = df_unique["dist_ukr"].max()
min_dist = df_unique["dist_ukr"].min()
df_unique["treat_ukr_norm"] = 1 - (df_unique["dist_ukr"] - min_dist) / (max_dist - min_dist)

# c) Gaussian decay
sigma = 1000
df_unique["treat_ukr_gauss"] = np.exp(-(df_unique["dist_ukr"] ** 2) / (2 * sigma ** 2))

# 13. --- Continuous treatment variables based on dist_invasion (with '2' suffix) ---

# a) Inverse-distance
df_unique["treat_invasion_inv"] = 1 / (1 + df_unique["dist_invasion"])

# b) Normalized distance (rescaled 0–1)
max_dist2 = df_unique["dist_invasion"].max()
min_dist2 = df_unique["dist_invasion"].min()
df_unique["treat_invasion_norm"] = 1 - (df_unique["dist_invasion"] - min_dist2) / (max_dist2 - min_dist2)

# c) Gaussian decay
sigma2 = 1000
df_unique["treat_invasion_gauss"] = np.exp(-(df_unique["dist_invasion"] ** 2) / (2 * sigma2 ** 2))

# 14. Save cleaned dataset (keeping all other variables)
df_unique.to_csv("sample.csv", index=False)

print(f"\nSaved {len(df_unique)} rows and {df_unique.shape[1]} columns to 'sample.csv'")